# Load Gold to Azure SQL
## Pipeline Monitor — Databricks Notebook 03

This notebook reads Gold Delta tables from ADLS
and loads them into Azure SQL (pipeline-gold database)
for querying via the monitoring dashboard and API.

### Tables Loaded:
1. dim_customers
2. fct_orders
3. mart_segment_revenue
4. mart_customer_ltv
5. mart_customer_activity

Configuration

In [0]:
# Cell 1 — Configuration
from pyspark.sql import functions as F
from datetime import datetime

# Read secrets from Databricks Secret Scope
STORAGE_ACCOUNT = dbutils.secrets.get(scope="pipeline-monitor", key="adls-account-name")
STORAGE_KEY     = dbutils.secrets.get(scope="pipeline-monitor", key="adls-account-key")
SQL_SERVER      = dbutils.secrets.get(scope="pipeline-monitor", key="sql-server")
SQL_PASSWORD    = dbutils.secrets.get(scope="pipeline-monitor", key="sql-password")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    STORAGE_KEY
)

GOLD = f"wasbs://gold@{STORAGE_ACCOUNT}.blob.core.windows.net"

# JDBC connection string for Azure SQL
JDBC_URL = (
    f"jdbc:sqlserver://{SQL_SERVER}:1433;"
    f"database=pipeline-gold;"
    f"encrypt=true;trustServerCertificate=false;"
    f"hostNameInCertificate=*.database.windows.net;"
    f"loginTimeout=30;"
)

SQL_PROPERTIES = {
    "user":     "sqladmin",
    "password": SQL_PASSWORD,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print(f"✅ Storage configured: {STORAGE_ACCOUNT}")
print(f"✅ SQL Server: {SQL_SERVER}")
print(f"✅ Gold path: {GOLD}")

## Step 1: Load Helper Function

A reusable function that reads a Gold Delta table
and writes it to Azure SQL using JDBC connector.

Mode: overwrite — replaces existing data each run
This ensures the SQL tables always match the Gold Delta tables.

In [0]:
# Cell 2 — Helper function to load Gold → Azure SQL
def load_to_sql(table_name: str, mode: str = "overwrite"):
    """
    Reads a Gold Delta table from ADLS and writes to Azure SQL.
    
    Args:
        table_name: name of the Gold table (e.g. 'fct_orders')
        mode: 'overwrite' replaces all data, 'append' adds to existing
    """
    print(f"  Loading {table_name}...")
    
    # Read from Gold Delta
    df = spark.read.format("delta").load(f"{GOLD}/{table_name}/")
    row_count = df.count()
    
    # Write to Azure SQL
    (df.write
       .mode(mode)
       .jdbc(url=JDBC_URL,
             table=f"dbo.{table_name}",
             properties=SQL_PROPERTIES))
    
    print(f"  ✅ {table_name}: {row_count} rows loaded to Azure SQL")
    return row_count

print("✅ Load function ready")

## Step 2: Load All Gold Tables to Azure SQL

Loads all 5 Gold tables into the pipeline-gold database.
Each table is overwritten on every run to keep SQL in sync
with the Gold Delta tables in ADLS.

In [0]:
# Cell 3 — Load all Gold tables to Azure SQL
print("\n=== Loading Gold Tables to Azure SQL ===\n")

total_rows = 0

tables = [
    "dim_customers",
    "fct_orders",
    "mart_segment_revenue",
    "mart_customer_ltv",
    "mart_customer_activity",
]

for table in tables:
    try:
        rows = load_to_sql(table)
        total_rows += rows
    except Exception as e:
        print(f"  ❌ {table} failed: {e}")

print(f"\n=== Complete: {total_rows} total rows loaded to Azure SQL ===")
dbutils.notebook.exit("SUCCESS")